# Getting a Resolved Schema

Before evaluation, you need a **resolved schema** — a simplified JSON Schema with only
`type`, `properties`, and `items`. No `$ref`, no `allOf`, no `anyOf`, no constraints.

Two ways to get one:
- **Option A:** Infer from gold data (`infer_schema`)
- **Option B:** Simplify an existing JSON Schema (`resolve_schema_references`)

This notebook shows both and compares the results.

## Option A: Infer from Gold Data

Your gold instances already resolve all ambiguity. `infer_schema()` reads the actual data and produces a clean schema.

In [2]:
import json
from struct_extract_eval import infer_schema

gold = [
    {"method": "sputtering", "temperature": 300, "lab_id": "A1"},
    {"method": "evaporation", "temperature": 450},
]

resolved_a = infer_schema(gold)
print(json.dumps(resolved_a, indent=2))

{
  "type": "object",
  "properties": {
    "lab_id": {
      "type": "string"
    },
    "method": {
      "type": "string"
    },
    "temperature": {
      "type": "integer"
    }
  }
}


`lab_id` is absent in the second instance, so it only appears when present.
`method` and `temperature` appear in every instance.

The result is clean: just `type` and `properties`. Ready for `annotate_xeval()`.

## Option B: Resolve an Existing JSON Schema

Real-world JSON Schemas are often complex — `$ref`, `allOf`, `anyOf`, `oneOf`,
constraints (`enum`, `minLength`, `format`), etc. The evaluator doesn't need any of that.

`resolve_schema_references()` simplifies it:

In [3]:
from struct_extract_eval import resolve_schema_references

# A typical complex schema with $ref, allOf, anyOf, constraints
complex_schema = {
    "$defs": {
        "Temperature": {
            "type": "object",
            "properties": {
                "value": {
                    "anyOf": [
                        {"exclusiveMinimum": 0, "type": "number"},
                        {"type": "null"},
                    ],
                    "default": None,
                    "title": "Value",
                },
                "unit": {
                    "type": "string",
                    "enum": ["K", "C", "F"],
                    "default": "K",
                },
            },
        },
    },
    "type": "object",
    "properties": {
        "method": {
            "type": "string",
            "title": "Synthesis Method",
            "minLength": 1,
        },
        "temperature": {"$ref": "#/$defs/Temperature"},
        "lab_id": {
            "allOf": [
                {"type": "string"},
                {"pattern": "^LAB-[0-9]+$"},
            ],
        },
    },
}

resolved_b = resolve_schema_references(complex_schema)
print(json.dumps(resolved_b, indent=2))

resolve_schema_references handles $ref, allOf, and anyOf[type, null].
The following are NOT handled and may cause issues:
  - oneOf: type info lost, fields may get wrong comparator ('exact' instead of 'numeric')
  - if/then/else: conditional properties lost, some fields won't be scored
  - anyOf with multiple non-null types: same risk as oneOf
Review the resolved output before passing to annotate_xeval.


{
  "type": "object",
  "properties": {
    "method": {
      "type": "string",
      "title": "Synthesis Method",
      "minLength": 1
    },
    "temperature": {
      "type": "object",
      "properties": {
        "value": {
          "default": null,
          "title": "Value",
          "exclusiveMinimum": 0,
          "type": "number"
        },
        "unit": {
          "type": "string",
          "enum": [
            "K",
            "C",
            "F"
          ],
          "default": "K"
        }
      }
    },
    "lab_id": {
      "type": "string",
      "pattern": "^LAB-[0-9]+$"
    }
  }
}


What happened:
- `$ref: "#/$defs/Temperature"` → resolved inline
- `allOf: [{type: string}, {pattern: ...}]` → merged into one dict
- `anyOf: [number, null]` → simplified to just `number`
- `$defs` → removed
- Constraints (`enum`, `default`, `title`, `minLength`, `pattern`, `exclusiveMinimum`) → left in but **ignored by the evaluator**

The warning tells you about keywords that are NOT handled (`oneOf`, `if/then/else`).
In this example there are none, so the resolved schema is complete.

## When Option B Can't Fully Resolve

If your schema uses `oneOf` or `if/then/else`, the type info inside those branches is lost.
The warning from `resolve_schema_references` tells you this.

**Example:** a field with `oneOf`

In [4]:
tricky_schema = {
    "type": "object",
    "properties": {
        "temperature": {
            "oneOf": [
                {"type": "number"},
                {"type": "string"},
            ]
        },
    },
}

resolved_tricky = resolve_schema_references(tricky_schema)
print(json.dumps(resolved_tricky, indent=2))
print()
print("Notice: 'temperature' still has 'oneOf' -- no 'type' key.")
print("annotate_xeval will default to 'exact' comparator (wrong for a number).")
print("For schemas like this, Option A (infer from gold) is safer.")

resolve_schema_references handles $ref, allOf, and anyOf[type, null].
The following are NOT handled and may cause issues:
  - oneOf: type info lost, fields may get wrong comparator ('exact' instead of 'numeric')
  - if/then/else: conditional properties lost, some fields won't be scored
  - anyOf with multiple non-null types: same risk as oneOf
Review the resolved output before passing to annotate_xeval.


{
  "type": "object",
  "properties": {
    "temperature": {
      "oneOf": [
        {
          "type": "number"
        },
        {
          "type": "string"
        }
      ]
    }
  }
}

Notice: 'temperature' still has 'oneOf' -- no 'type' key.
annotate_xeval will default to 'exact' comparator (wrong for a number).
For schemas like this, Option A (infer from gold) is safer.


## Next Step: Annotate with Eval Defaults

Once you have a resolved schema (from either option), add eval defaults and start evaluating.
See `01_example_simple.ipynb`.

```python
from struct_extract_eval import annotate_xeval

annotate_xeval(resolved_schema)  # adds x-eval-compare to each leaf field
# save to JSON, review, then evaluate
```